# 02 spaCy NER — Entrevistas reales

🎯 **Objetivo:** Identificar entidades candidatas (personas, lugares, organizaciones, y referencias sociodemográficas) en las entrevistas usando spaCy, evaluar la calidad de los resultados y detectar patrones problemáticos antes de pasarlos al LLM o GLiNER.

🔍 **Puntos clave:**
1. Carga de entrevistas desde `data/raw/entrevistas_originales/`
2. Estadísticas básicas del corpus
3. Extracción de entidades con SpacyNER
4. Visualización de resultados
5. Revisión manual de entidades
6. Análisis de frecuencia de entidades
7. Detección de problemas (falsos positivos evidentes, omisiones)
8. Exportar candidatos a JSON para merge + filter antes de pasar al LMM

## 0 · Imports y configuración

In [ ]:
import sys
import json
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd
from spacy import displacy
from IPython.display import display, HTML

# Agrega la raíz del proyecto al path para importar src/
PROJECT_ROOT = Path("../").resolve()  # ajusta si se corre desde otro lugar
sys.path.insert(0, str(PROJECT_ROOT))

from src.ner.spacy_ner import SpacyNER, DocumentResult

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
DATA_DIR      = PROJECT_ROOT / "data" / "raw" / "entrevistas_originales"
OUTPUT_DIR    = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_spacy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros ─────────────────────────────────────────────────────────────
SPACY_MODEL   = "es_core_news_lg"   # cambia a es_core_news_sm si hay problemas de memoria
ENCODING      = "utf-8"             # cambia a "latin-1" si TXT tienen problemas de tildes

print(f"Datos:   {DATA_DIR}")
print(f"Salida:  {OUTPUT_DIR}")

## 1 · Carga de entrevistas

In [ ]:
def load_interviews(data_dir: Path, encoding: str = "utf-8") -> dict[str, str]:
    """
    Carga todos los .txt de data_dir.
    Devuelve {nombre_archivo_sin_extension: texto}.
    """
    interviews = {}
    txt_files = sorted(data_dir.glob("*.txt"))

    if not txt_files:
        print(f"⚠️  No se encontraron archivos .txt en:\n   {data_dir}")
        return interviews

    for path in txt_files:
        try:
            text = path.read_text(encoding=encoding).strip()
            interviews[path.stem] = text
            print(f"  ✓ {path.name:<40} {len(text.split()):>7,} palabras")
        except UnicodeDecodeError:
            print(f"  ✗ {path.name} — error de encoding, prueba ENCODING='latin-1'")

    print(f"\nTotal: {len(interviews)} entrevistas cargadas.")
    return interviews

In [ ]:
interviews = load_interviews(DATA_DIR, encoding=ENCODING)

## 2 · Estadísticas básicas del corpus

In [ ]:
def corpus_stats(interviews: dict[str, str]) -> pd.DataFrame:
    rows = []
    for doc_id, text in interviews.items():
        words  = len(text.split())
        chars  = len(text)
        lines  = text.count("\n") + 1
        rows.append({"entrevista": doc_id, "palabras": words,
                     "caracteres": chars, "lineas": lines})
    return pd.DataFrame(rows).set_index("entrevista")

In [ ]:
stats = corpus_stats(interviews)
display(stats)
print(f"\nTotal palabras: {stats['palabras'].sum():,}")
print(f"Promedio por entrevista: {stats['palabras'].mean():,.0f} palabras - {stats['caracteres'].mean():,.0f} caracteres")

## 3 · Inicializar SpacyNER y procesar todas las entrevistas

In [ ]:
ner = SpacyNER(model=SPACY_MODEL)
print(ner.model_info())

In [ ]:
# Procesar en lote
doc_ids = list(interviews.keys())
texts   = list(interviews.values())

In [ ]:
print("Procesando con spaCy…")
results = ner.process_batch(texts, doc_ids=doc_ids)

In [ ]:
# Resumen rápido
for r in results:
    print(f"  {r.doc_id:<40} → {len(r.entities):>3} entidades")

In [ ]:
# Resumen rápido
print(f"\n✅ Listo. Total entidades: {sum(len(r.entities) for r in results)}")

## 4 · Visualización de entidades en el texto

Usamos `displacy` de spaCy para ver exactamente qué detectó en cada entrevista.

In [ ]:
# ── Parámetros de visualización ────────────────────────────────────────────
INTERVIEW_TO_SHOW = doc_ids[3]   # ← cambia el índice para ver otra entrevista
MAX_CHARS_DISPLAY = None         # recorta para no saturar la pantalla

# Colores por tipo de entidad (para displacy)
COLORS = {
    "PER":  "#ffd6d6",   # rosa · personas
    "LOC":  "#d6f0ff",   # azul · lugares
    "ORG":  "#d6ffd9",   # verde · organizaciones
    # "MISC": "#fff3d6",   # amarillo
}
OPTIONS = {"ents": ["PER", "LOC", "ORG"], "colors": COLORS}

In [ ]:
# Procesar solo el fragmento para visualizar, si es None es toda
text_snippet = interviews[INTERVIEW_TO_SHOW][:MAX_CHARS_DISPLAY] if MAX_CHARS_DISPLAY is not None else interviews[INTERVIEW_TO_SHOW]

In [ ]:
doc = ner.nlp(text_snippet)

In [ ]:
print(f"\nMostrando: {INTERVIEW_TO_SHOW} (primeros {MAX_CHARS_DISPLAY} caracteres)\n")
display(HTML(displacy.render(doc, style="ent", options=OPTIONS, jupyter=True)))

## 5 · Revisión manual por entrevista

Selecciona una entrevista y ve todas sus entidades con contexto.

In [ ]:
# ── Cambia este valor para inspeccionar cada entrevista ───────────────────
REVISAR = INTERVIEW_TO_SHOW

In [ ]:
result = next(r for r in results if r.doc_id == REVISAR)
ner.print_entities(result)

## 6 · Análisis de frecuencia de entidades

In [ ]:
def build_entity_dataframe(results: list[DocumentResult]) -> pd.DataFrame:
    """Convierte todos los resultados en un DataFrame plano."""
    rows = []
    for r in results:
        for ent in r.entities:
            rows.append({
                "entrevista": r.doc_id,
                "texto":      ent.text,
                "texto_norm": ent.text.lower().strip(),  # para agrupar variantes
                "tipo":       ent.label,
                "inicio":     ent.start,
                "fin":        ent.end,
            })
    return pd.DataFrame(rows)

In [ ]:
df = build_entity_dataframe(results)

In [ ]:
df.sample(n=5, random_state=42)

In [ ]:
print(f"Total de menciones: {len(df)}")
print(f"\nDistribución por tipo:")
display(df["tipo"].value_counts().rename("menciones").to_frame())

In [ ]:
# Entidades más frecuentes por tipo
for tipo in ["PERSONA", "LUGAR", "ORGANIZACION"]:
    subset = df[df["tipo"] == tipo]
    if subset.empty:
        print(f"\n── {tipo}: ninguna ──")
        continue
    top = (
        subset.groupby("texto_norm")["texto"]
        .agg(menciones="count")
        .sort_values("menciones", ascending=False)
        .head(15)
    )
    print(f"\n── Top {tipo} ──")
    display(top)

In [ ]:
# Entidades que aparecen en MÁS DE UNA entrevista
# (pueden ser personas que se repiten entre entrevistados → importante para el diccionario)
cross_doc = (
    df.groupby(["texto_norm", "tipo"])["entrevista"]
    .nunique()
    .reset_index()
    .rename(columns={"entrevista": "n_entrevistas"})
    .query("n_entrevistas > 1")
    .sort_values("n_entrevistas", ascending=False)
)

In [ ]:
print("Entidades que aparecen en más de una entrevista:")
display(cross_doc if not cross_doc.empty else "Ninguna")

## 7 · Detección de problemas

Esta sección ayuda a identificar **falsos positivos evidentes** y **posibles omisiones** antes de pasar al LLM.

In [ ]:
# ── 6.1 Entidades sospechosas de ser falsos positivos ─────────────────────
# Heurísticas: palabras muy cortas, solo números, y/o palabras comunes del área

PALABRAS_AREA = {
    "maíz", "milpa", "frijol", "tortilla", "campo", "siembra", "cosecha",
    "gobierno", "programa", "apoyo", "proyecto", "comunidad", "municipio",
    "señor", "señora", "familia", "esposo", "esposa", "hijo", "hija",
}

In [ ]:
def es_sospechoso(texto: str) -> str | None:
    t = texto.strip().lower()
    if len(t) <= 2:
        return "muy corta"
    if re.fullmatch(r"[\d\s]+", t):
        return "solo números"
    if t in PALABRAS_AREA:
        return "palabra del área de seguridad alimentaria"
    return None

In [ ]:
df["sospecha"] = df["texto"].apply(es_sospechoso)
sospechosos = df[df["sospecha"].notna()]

In [ ]:
print(f"Entidades sospechosas: {len(sospechosos)} de {len(df)} ({100*len(sospechosos)/max(len(df),1):.1f}%)")
display(sospechosos[["entrevista", "texto", "tipo", "sospecha"]].head(30))

In [ ]:
# ── 6.2 Contexto de entidades sospechosas ─────────────────────────────────
# Ver las N palabras alrededor para juzgar si es FP o no

WINDOW = 40  # caracteres de contexto a cada lado

def show_context(row: pd.Series, interviews: dict[str, str]) -> str:
    text = interviews.get(row["entrevista"], "")
    start = max(0, row["inicio"] - WINDOW)
    end   = min(len(text), row["fin"] + WINDOW)
    snippet = text[start:end].replace("\n", " ")
    # Resalta la entidad
    ent_text = row["texto"]
    snippet = snippet.replace(ent_text, f">>>{ent_text}<<<")
    return snippet

In [ ]:
if not sospechosos.empty:
    print("Contexto de las primeras 10 entidades sospechosas:\n")
    for _, row in sospechosos.head(10).iterrows():
        ctx = show_context(row, interviews)
        print(f"[{row['tipo']}] '{row['texto']}' ({row['sospecha']})")
        print(f"  …{ctx}…")
        print()

In [ ]:
# ── 6.3 Entidades únicas por entrevista (aparecen solo 1 vez) ─────────────
# Las menciones únicas son las más difíciles de anonimizar consistentemente
# El LLM necesitará más atención sobre ellas

menciones_por_doc = df.groupby(["entrevista", "texto_norm"]).size().reset_index(name="count")
hapax = menciones_por_doc[menciones_por_doc["count"] == 1]

In [ ]:
print(f"Entidades con una sola mención: {len(hapax)}")
print(f"(El LLM deberá decidir si son reales o falsos positivos)\n")
display(hapax.head(20))

## 8 · Exportar candidatos a JSON

Genera un JSON por entrevista en `data/processed/entidades_candidatas_spacy/`.  
Antes de enviar al LLM (Ollama), se filtrarán las entidades y se mezcalarán, y ese es el input que recibirá el LLM.

In [ ]:
def export_candidates(results: list[DocumentResult], output_dir: Path) -> None:
    """
    Exporta cada DocumentResult como JSON individual.
    Formato:
    {
      "doc_id": "entrevista_01",
      "text": "...",
      "spacy_candidates": [
        {"text": "Ana", "label": "PERSONA", "start": 12, "end": 15, ...},
        ...
      ],
      "stats": {"total": 5, "PERSONA": 2, "LUGAR": 2, "ORGANIZACION": 1}
    }
    """
    for result in results:
        # Conteo por tipo
        type_counts: dict[str, int] = defaultdict(int)
        for ent in result.entities:
            type_counts[ent.label] += 1

        payload = {
            "doc_id":            result.doc_id,
            "text":              result.text,
            "spacy_candidates":  [e.to_dict() for e in result.entities],
            "stats": {
                "total":         len(result.entities),
                **dict(type_counts),
            },
        }

        out_path = output_dir / f"{result.doc_id}_candidates.json"
        out_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=4),
            encoding="utf-8",
        )
        print(f"  ✓ {out_path.name}  ({len(result.entities)} entidades)")

    # También exporta un resumen consolidado de todo el corpus
    summary = {
        "ner_summary": {
            "n_documents":    len(results),
            "total_entities": sum(len(r.entities) for r in results),
            "by_document":    [
                {"doc_id": r.doc_id, "n_entities": len(r.entities)}
                for r in results
            ],
        }
    }
    summary_path = output_dir / "summary_spacy.json"
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=4), encoding="utf-8"
    )
    print(f"\n  ✓ Resumen: {summary_path.name}")

In [ ]:
print(f"Exportando a: {OUTPUT_DIR}\n")
export_candidates(results, OUTPUT_DIR)

## 9 · Resumen final y notas para el LLM

Genera un reporte de lo que spaCy encontró y los patrones que el LLM deberá resolver.

In [ ]:
print("=" * 60)
print("RESUMEN — SpaCy NER sobre corpus completo")
print("=" * 60)

total_ents = sum(len(r.entities) for r in results)
print(f"\n  Entrevistas procesadas : {len(results)}")
print(f"  Entidades encontradas  : {total_ents}")

print(f"\n  Por tipo:")
for tipo, count in df["tipo"].value_counts().items():
    print(f"    {tipo:<20} {count:>4}")

print(f"\n  Posibles FP detectados : {len(sospechosos)}")
print(f"  Entidades únicas (hapax): {len(hapax)}")

print(f"\n  Archivos JSON generados en:")
print(f"    {OUTPUT_DIR}")

print(f"""
─────────────────────────────────────────────────────
TAREAS PARA EL SIGUIENTE PASO:
  1. Confirmar entidades reales vs falsos positivos
  2. Agregar entidades omitidas por spaCy
     (apodos, roles: 'el dueño del rancho', 'mi cuñada')
  3. Construir diccionario coherente:
     'Ana', 'Ana García', 'doña Ana' → persona_1
─────────────────────────────────────────────────────
""")